In [91]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from google.colab import drive
from collections import Counter

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [92]:
def load_and_scale_data(filepath):
    """
    Loads car data from a CSV file, extracts features and target,
    and applies MinMax scaling to the features.
    """
    df = pd.read_csv(filepath)

    original_features = df[["Volume", "Doors"]]
    target = df["Style"]

    scaler = MinMaxScaler()
    scaled_array = scaler.fit_transform(original_features)
    features_scaled = pd.DataFrame(scaled_array, columns=["Volume", "Doors"])

    return features_scaled, original_features, target

In [93]:
features_scaled, original_features, target = load_and_scale_data(
    "/content/drive/MyDrive/Colab Notebooks/AllCars.csv"
)

In [94]:
processed_df = features_scaled.copy()
processed_df["Style"] = target

In [95]:
kmeans = KMeans(n_clusters=5, random_state=3)
processed_df["Cluster"] = kmeans.fit_predict(features_scaled)


In [96]:
cluster_style_map = {}

for cluster_id in range(5):
    cluster_data = processed_df[processed_df["Cluster"] == cluster_id]
    majority_style = Counter(cluster_data["Style"]).most_common(1)[0][0]
    cluster_style_map[cluster_id] = majority_style

processed_df["ClusterStyle"] = processed_df["Cluster"].map(cluster_style_map)


In [97]:
cluster_cars = original_features.copy()
cluster_cars["Style"] = target.values
cluster_cars["ClusterStyle"] = processed_df["Cluster"].map(cluster_style_map).values
cluster_cars.to_csv("ClusterCars.csv", index=False)
print("Created ClusterCars.csv")

Created ClusterCars.csv


In [100]:
accuracy_rows = []

for cluster_id in range(5):
    cluster_data = processed_df[processed_df["Cluster"] == cluster_id]
    size = len(cluster_data)
    majority_style = cluster_style_map[cluster_id]
    correct = len(cluster_data[cluster_data["Style"] == majority_style])
    accuracy = correct / size if size > 0 else 0

    accuracy_rows.append([
        majority_style,
        size,
        accuracy
    ])

cluster_accuracy = pd.DataFrame(
    accuracy_rows,
    columns=["ClusterStyle", "SizeOfCluster", "Accuracy"]
)

cluster_accuracy.to_csv("ClusterAccuracy.csv", index=False)
print("Created ClusterAccuraccy.csv")

Created ClusterAccuraccy.csv
